# Intake triage walkthrough

Logic lives in `intake_triage`. This notebook only imports it.
Hartwell Grey / Google Sheets are assumptions, not in the brief.


In [ ]:
from intake_triage.generate import seed_to_enquiry, seed_to_extraction
from intake_triage.pipeline import triage_from_extraction
from intake_triage.emit import format_email, sheet_row
from intake_triage.policy_loader import load_policy
from intake_triage.seeds import SEEDS

policy = load_policy()
by_id = {s["enquiry_id"]: s for s in SEEDS}
clean, stacked, ambiguous = (by_id[i] for i in ("HG-2026-0001", "HG-2026-0008", "HG-2026-0011"))
print(clean["company_name"], stacked["company_name"], ambiguous["company_name"])


In [ ]:
def show_drivers(seed):
    ext = seed_to_extraction(seed)
    print(seed["enquiry_id"])
    for sig in ext.work_signals:
        print(f"  signal={sig.value} span={sig.evidence_span!r}")
    print("  jurs", ext.jurisdiction_names.value, ext.jurisdiction_names.evidence_span)

show_drivers(clean)
show_drivers(stacked)
show_drivers(ambiguous)


In [ ]:
for seed in (clean, stacked, ambiguous):
    d = triage_from_extraction(seed_to_enquiry(seed), seed_to_extraction(seed), policy)
    print(seed["enquiry_id"], "hours", d.estimated_hours, "tier", d.complexity)
    for line in d.rule_trace:
        print(" ", line)
    print()


In [ ]:
for seed in (clean, stacked, ambiguous):
    d = triage_from_extraction(seed_to_enquiry(seed), seed_to_extraction(seed), policy)
    print(seed["enquiry_id"], "abstained", d.abstained, "route", d.route_to, "reason", d.abstain_reason)
    if d.abstained:
        for c in d.competing_lines:
            print("  candidate", c.service_line, c.hours, c.owner)


In [ ]:
from intake_triage.evaluate import run_eval
print(run_eval())


In [ ]:
enq = seed_to_enquiry(clean)
dec = triage_from_extraction(enq, seed_to_extraction(clean), policy)
print(format_email(enq, dec, policy))
print(sheet_row(enq, dec))
